<a href="https://colab.research.google.com/github/Lazyyy-Panda/BIS/blob/main/BIS_Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Title: Constraint-Handling in Genetic Algorithms Through the Use of Dominance-Based Tournament Selection  
 Authors: Carlos A. Coello Coello and Efrén Mezura-Montes   
 Journal: IEEE Transactions on Evolutionary Computation

***Problem Statement:***
The goal is to design a helical tension/compression spring such that its overall mass/weight is minimized while satisfying constraints on shear stress, surge frequency, minimum deflection, and physical outer diameter limits.

Decision Variables

The design is defined by a 3-dimensional vector $\vec{x} = [x_1, x_2, x_3]^T$:$x_1 = d$: Wire diameter$x_2 = D$: Mean coil diameter$x_3 = N$: Number of active coils

Objective Function (Minimize Weight)$$f(\vec{x}) = (x_3 + 2) x_2 x_1^2$$Inequality Constraints ($g_i(\vec{x}) \le 0$)Deflection Constraint:$$g_1(\vec{x}) = 1 - \frac{x_2^3 x_3}{71785 x_1^4} \le 0$$Shear Stress Constraint:$$g_2(\vec{x}) = \frac{4 x_2^2 - x_1 x_2}{12566 (x_2 x_1^3 - x_1^4)} + \frac{1}{5108 x_1^2} - 1 \le 0$$Surge Frequency Constraint:$$g_3(\vec{x}) = 1 - \frac{140.45 x_1}{x_2^2 x_3} \le 0$$Outside Diameter Limits Constraint:$$g_4(\vec{x}) = \frac{x_1 + x_2}{1.5} - 1 \le 0$$Variable Bounds$0.05 \le x_1 \le 2.00$$0.25 \le x_2 \le 1.30$$2.00 \le x_3 \le 15.00$

GA Objective-to-Fitness ConversionSince your Genetic Algorithm evaluates fitness for a maximization problem (MaxSolution), you map the cost function $f(\vec{x})$ and constraint violations into a fitness value $F(\vec{x})$:Constraint Violation Penalty:$$P(\vec{x}) = C \sum_{i=1}^{4} \left( \max(0, g_i(\vec{x})) \right)^2$$(where $C = 10^5$ or a similar large penalty coefficient)Fitness Function:$$\text{Fitness}(\vec{x}) = \frac{1}{f(\vec{x}) + P(\vec{x})}$$

Algorithm geneticAlgo:
  Input: PopulationSize, MaxGenerations, CrossoverRate, MutationRate
  Output: MaxSolution
  1. Initialize population randomly with 'PopulationSize' candidate solution
  2. FOR generation=1 to MaxGenerations:

     a. Evaluate Fitness:
        FOR each individual in the population:
          Calculate fitness= ObjectiveFunction(individual)






    
    
    
     b. Selection:
        Select pairs of parents biased towards higher fitness

     c.Crossover:
     FOR each parent pair:
        IF random() < CrossoverRate :
           child1,child2 = Combine(parent1,parent2)
        ELSE:
           child1, child2= Copy(parent1,parent2)

     d. Mutation:
        FOR each child:
           IF random() < MutationRate:
              Apply small random change to child

     e. Replace old population with the new generation of children
     
     f. Track and record the individual with the highest fitness so far

  3. Return MaxSolution

In [7]:
import random

# ==========================================
# PROBLEM DEFINITION: Tension/Compression Spring
# ==========================================

# Bounds for variables: x1 (d), x2 (D), x3 (N)
BOUNDS = [
    (0.05, 2.00),   # x1: wire diameter (d)
    (0.25, 1.30),   # x2: mean coil diameter (D)
    (2.00, 15.00)   # x3: number of active coils (N)
]

def objective_function(x):
    """Calculate spring weight f(x)."""
    x1, x2, x3 = x
    return (x3 + 2.0) * x2 * (x1 ** 2)

def constraints(x):
    """Return inequality constraint values g_i(x) <= 0."""
    x1, x2, x3 = x

    # Avoid division by zero or invalid domain for x2 <= x1
    if x2 <= x1 or x1 <= 0 or x2 <= 0 or x3 <= 0:
        return [1e6, 1e6, 1e6, 1e6]

    g1 = 1.0 - (x2**3 * x3) / (71785.0 * (x1**4))

    denom_g2 = 12566.0 * (x2 * (x1**3) - (x1**4))
    if denom_g2 == 0:
        g2 = 1e6
    else:
        g2 = (4.0 * x2**2 - x1 * x2) / denom_g2 + (1.0 / (5108.0 * (x1**2))) - 1.0

    g3 = 1.0 - (140.45 * x1) / ((x2**2) * x3)
    g4 = (x1 + x2) / 1.5 - 1.0

    return [g1, g2, g3, g4]

def calculate_fitness(x):
    """
    Evaluates Fitness = ObjectiveFunction(individual) mapped to maximization.
    Applies penalty for constraint violations.
    """
    weight = objective_function(x)
    g_vals = constraints(x)

    # Quadratic penalty for violating g_i(x) > 0
    penalty = sum(max(0.0, g)**2 for g in g_vals)
    penalty_coefficient = 1e6

    total_cost = weight + penalty_coefficient * penalty

    # Maximization objective: lower total cost yields higher fitness
    return 1.0 / total_cost

# ==========================================
# GENETIC ALGORITHM OPERATORS
# ==========================================

def initialize_individual():
    """Step 1: Random candidate solution within variable bounds."""
    return [random.uniform(lb, ub) for lb, ub in BOUNDS]

def selection(population, fitnesses, k=3):
    """Step 2b: Tournament selection biased towards higher fitness."""
    selected_indices = random.sample(range(len(population)), k)
    best_index = max(selected_indices, key=lambda idx: fitnesses[idx])
    return population[best_index]

def crossover(parent1, parent2, crossover_rate):
    """Step 2c: Arithmetic Crossover."""
    if random.random() < crossover_rate:
        alpha = random.random()
        child1 = [alpha * p1 + (1.0 - alpha) * p2 for p1, p2 in zip(parent1, parent2)]
        child2 = [(1.0 - alpha) * p1 + alpha * p2 for p1, p2 in zip(parent1, parent2)]
        return child1, child2
    else:
        return list(parent1), list(parent2)

def mutate(individual, mutation_rate):
    """Step 2d: Gaussian Mutation with boundary clamping."""
    mutated = list(individual)
    for i in range(len(mutated)):
        if random.random() < mutation_rate:
            lb, ub = BOUNDS[i]
            std_dev = (ub - lb) * 0.1  # 10% of domain range
            mutated[i] += random.gauss(0, std_dev)
            # Clamp to bounds
            mutated[i] = max(lb, min(ub, mutated[i]))
    return mutated

# ==========================================
# MAIN ALGORITHM FUNCTION
# ==========================================

def geneticAlgo(population_size=100, max_generations=500, crossover_rate=0.8, mutation_rate=0.15):
    """
    Main Genetic Algorithm procedure matching user pseudocode structure.
    """
    # 1. Initialize population randomly
    population = [initialize_individual() for _ in range(population_size)]

    best_solution = None
    best_fitness = -float('inf')

    # 2. Loop for generations
    for generation in range(1, max_generations + 1):

        # a. Evaluate Fitness
        fitnesses = [calculate_fitness(ind) for ind in population]

        # f. Track and record individual with highest fitness so far
        for ind, fit in zip(population, fitnesses):
            if fit > best_fitness:
                best_fitness = fit
                best_solution = list(ind)

        new_population = []

        # Loop to produce new generation via pairs
        while len(new_population) < population_size:
            # b. Selection
            parent1 = selection(population, fitnesses)
            parent2 = selection(population, fitnesses)

            # c. Crossover
            child1, child2 = crossover(parent1, parent2, crossover_rate)

            # d. Mutation
            child1 = mutate(child1, mutation_rate)
            child2 = mutate(child2, mutation_rate)

            new_population.extend([child1, child2])

        # Truncate if excess children created
        new_population = new_population[:population_size]

        # e. Replace old population with new generation
        population = new_population

    # 3. Return MaxSolution
    return best_solution, best_fitness

# ==========================================
# EXECUTION AND OUTPUT
# ==========================================

if __name__ == "__main__":
    random.seed(42)  # For reproducible results

    # Run Algorithm
    best_x, best_fit = geneticAlgo(
        population_size=120,
        max_generations=1000,
        crossover_rate=0.85,
        mutation_rate=0.20
    )

    # Evaluate final metrics
    best_weight = objective_function(best_x)
    constraint_violations = constraints(best_x)

    print("==============================================")
    print("   GENETIC ALGORITHM OPTIMIZATION RESULTS     ")
    print("==============================================")
    print(f"Optimal Wire Diameter (x1 = d)    : {best_x[0]:.5f}")
    print(f"Optimal Coil Diameter (x2 = D)    : {best_x[1]:.5f}")
    print(f"Optimal Active Coils  (x3 = N)    : {best_x[2]:.5f}")
    print("----------------------------------------------")
    print(f"Minimized Spring Weight f(x)       : {best_weight:.6f}")
    print(f"Maximized Fitness Score           : {best_fit:.6f}")
    print("----------------------------------------------")
    print("Constraint Violations g_i(x) <= 0:")
    for i, g in enumerate(constraint_violations, start=1):
        status = "Satisfied" if g <= 0 else "VIOLATED"
        print(f"  g{i}(x) = {g: .6f}  [{status}]")
    print("==============================================")

   GENETIC ALGORITHM OPTIMIZATION RESULTS     
Optimal Wire Diameter (x1 = d)    : 0.05684
Optimal Coil Diameter (x2 = D)    : 0.49284
Optimal Active Coils  (x3 = N)    : 6.36015
----------------------------------------------
Minimized Spring Weight f(x)       : 0.013312
Maximized Fitness Score           : 75.118982
----------------------------------------------
Constraint Violations g_i(x) <= 0:
  g1(x) = -0.015952  [Satisfied]
  g2(x) = -0.001674  [Satisfied]
  g3(x) = -4.167928  [Satisfied]
  g4(x) = -0.633548  [Satisfied]
